# RC9.2.1 — DEEP 14,400 s, run as resumable segments
## Day-2 follow-up (SAMPLE)

This is the **second** package. Run it only where the 2,700 s pass says a full
budget is needed, or on the scenarios you care most about.

### Why segments and not one 4-hour run

A four-hour unattended run does not survive a Colab disconnect. It did not
survive in our own container either — two attempts were killed at 10 and 40
minutes. So the budget is reached as **6 segments of 2,400 s**, each resuming
the last from the engine's own checkpoints. A lost segment costs one segment.

### Resume only started working today

`--resume` was broken in every RC9.2.1 build before this package. `run_id` was
hashed from `run_parameters`, which contained an absolute wall-clock timestamp,
so the recomputed identity never matched the checkpoint and resume always raised
*"Resume refused"*. That is fixed here (finding **A32**), and the engine in this
package is the fixed one — check `SCENARIOS.json` → `engine_sha256`.

**One scenario per Colab instance.** Do not shard scenarios inside one instance
at this budget.

### Variant B: with Google Drive — **recommended**

Results and checkpoints live on Drive, so a disconnect costs nothing: reconnect, re-run cells 1–3, and resume from the last segment.

### If the Drive mount really does hang

The usual causes, in order:

1. **The authorisation popup was blocked.** Allow popups for `colab.research.google.com` and re-run the cell.
2. **Third-party cookies are blocked.** Drive auth needs them. Allow them for Google domains, or use a normal (non-incognito) window.
3. **A stale mount.** Runtime → Restart runtime, then re-run. If it persists, change the mount call to `drive.mount("/content/drive", force_remount=True)`.

**If Drive is just being difficult, stop fighting it** — use the no-Drive notebook instead. You upload the 2 MB ZIP into each session and download the results at the end. For a 45-minute run that is perfectly fine; you only lose the protection against a mid-run disconnect.


In [ ]:
#@title 1. Environment { display-mode: "form" }
!pip -q install "ortools==9.15.6755" "openpyxl>=3.1" 2>&1 | tail -2
import multiprocessing, platform
print("cpus:", multiprocessing.cpu_count(), "| python:", platform.python_version())
import ortools; print("ortools:", ortools.__version__)

In [ ]:
#@title 2. Mount Drive and locate the ZIP { display-mode: "form" }
# ZIP_PATH is looked at FIRST. Only if it is missing do we search, and then only
# a few shallow locations - never the whole Drive.
#
# The previous version of this cell ran rglob() over /content/drive, which walks
# your ENTIRE Drive over a network filesystem. Every directory listing is a
# round trip, so on a Drive with real content the cell simply never returns and
# looks like the mount is hanging. It was not the mount.
ZIP_NAME = "RC9_2_1_DEEP_14400S_FOLLOWUP_SAMPLE.zip"  #@param {type:"string"}
ZIP_PATH = ""  #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")          # add force_remount=True if it misbehaves

import pathlib, itertools
MYDRIVE = pathlib.Path("/content/drive/MyDrive")

def find_zip(name: str):
    """Shallow, bounded search. Never walks the whole Drive."""
    seen = []
    # 1. MyDrive root, 2. a few usual folders, 3. one level down - and stop.
    roots = [MYDRIVE, MYDRIVE / "Colab Notebooks", MYDRIVE / "Downloads"]
    for root in roots:
        candidate = root / name
        if candidate.exists():
            return candidate, seen
        seen.append(str(root))
    try:
        for child in itertools.islice(sorted(p for p in MYDRIVE.iterdir() if p.is_dir()), 40):
            candidate = child / name
            if candidate.exists():
                return candidate, seen
            seen.append(str(child))
    except OSError:
        pass
    return None, seen

if ZIP_PATH.strip():
    found = pathlib.Path(ZIP_PATH.strip())
    assert found.exists(), f"ZIP_PATH does not exist: {found}"
else:
    found, looked = find_zip(ZIP_NAME)
    if found is None:
        print("Could not find", ZIP_NAME, "in:")
        for p in looked[:12]:
            print("   ", p)
        print("\nPut the ZIP in the top level of MyDrive, or paste its full path "
              "into ZIP_PATH above and re-run this cell.")
        raise SystemExit("ZIP not found")

ZIP_PATH = str(found)
print("using:", ZIP_PATH)

In [ ]:
#@title 3. Extract; checkpoints and results go to Drive { display-mode: "form" }
DRIVE_RESULTS = "/content/drive/MyDrive/RC921_DEEP_RESULTS"  #@param {type:"string"}
import zipfile, pathlib, os, json
os.makedirs("/content/rc921deep", exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall("/content/rc921deep")
c = list(pathlib.Path("/content/rc921deep").rglob("SCENARIOS.json"))
assert c, "SCENARIOS.json not found"
ROOT = c[0].parent
# On Drive, so a disconnect cannot lose banked segments.
RESULTS_ROOT = pathlib.Path(DRIVE_RESULTS); RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
m = json.loads((ROOT/"SCENARIOS.json").read_text())
print("engine:", m["engine_release"]); print("sha256:", m["engine_sha256"][:32], "…")
print("RESULTS_ROOT =", RESULTS_ROOT, "(Drive)")

In [ ]:
#@title 4. Run the segments { display-mode: "form" }
SCENARIO      = "CRICUT_VOICE"  #@param ["NMG_SP","CRICUT_VOICE","CRICUT_CHAT","AE_AR_B2B","GDI_REAL28","NMG_EN_SP","NMG_EN"]
SEGMENT_SEC   = 2400  #@param {type:"integer"}
SEGMENTS      = 6     #@param {type:"integer"}
CONTINUE_FROM = 1     #@param {type:"integer"}

import subprocess, sys, multiprocessing
cmd = [sys.executable, "-u", str(ROOT/"runners"/"rc921_deep_segmented.py"),
       "--package-root", str(ROOT), "--results-root", str(RESULTS_ROOT),
       "--only", SCENARIO, "--segment-sec", str(SEGMENT_SEC),
       "--segments", str(SEGMENTS), "--continue-from", str(CONTINUE_FROM),
       "--num-workers", str(multiprocessing.cpu_count())]
print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout: print(line, end="")
print("\nexit:", proc.wait())

## 5. If Colab disconnects mid-run

Re-open the notebook, re-run cells 1–3, then set **`CONTINUE_FROM`** to the
segment that did not finish and run cell 4 again. Completed segments are already
banked in the engine's checkpoints.

`<SCENARIO>_SEGMENT_LEDGER.json` in the results directory records which segments
ran and how many skeletons were banked after each — read it to find where to
resume.

With Drive the checkpoints survive the disconnect, so resuming genuinely continues rather than restarting.

In [ ]:
#@title 6. (optional) Zip the Drive results { display-mode: "form" }
import shutil, time, os
out = f"/content/drive/MyDrive/RC921_DEEP_{SCENARIO}_{time.strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(out, "zip", str(RESULTS_ROOT))
print("written:", out+".zip", round(os.path.getsize(out+".zip")/1e6,1), "MB")